# Data preprocessing pipeline (template)

This notebook is a **generic template** for common ML data preprocessing steps:
- Load raw data
- Identify target and feature columns
- Detect categorical vs numerical features
- Train/test split
- Build a preprocessing pipeline (One-Hot for categoricals; optional scaling/imputation)
- Transform data into model-ready matrices
- Save processed datasets and preprocessing artifacts

> Adapt the **paths**, **target column name**, and any dataset-specific cleaning rules.

In [ ]:
# =========================
# 0) Imports
# =========================
import os
import pickle
from pathlib import Path

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [ ]:
# =========================
# 1) Project paths (adjust)
# =========================
# If this notebook is inside notebooks/, the project root is one level up.
PROJECT_ROOT = Path.cwd().parents[0]

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

RAW_DATA_PATH = DATA_DIR / "bank.csv"             # <-- change as needed
PROCESSED_DIR = DATA_DIR / "processed"            # optional
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH

In [ ]:
# =========================
# 2) Load raw data
# =========================
df = pd.read_csv(RAW_DATA_PATH)
df.head()

In [ ]:
# =========================
# 3) Basic checks
# =========================
print("Shape:", df.shape)
print("\nDtypes (top 20):")
display(df.dtypes.head(20))

print("\nMissing values (top 20):")
display(df.isna().sum().sort_values(ascending=False).head(20))

## 4) Define target and features

Set the target column name below.  
If your dataset needs custom cleaning (e.g., mapping `yes/no` to `1/0`), do it here.

In [ ]:
# =========================
# 4) Target column
# =========================
TARGET_COLUMN = "deposit"  # <-- change as needed

# Example: map yes/no target to 1/0 (only if your dataset uses this pattern)
if TARGET_COLUMN in df.columns and df[TARGET_COLUMN].dtype == "object":
    mapping = {"yes": 1, "no": 0}
    if set(df[TARGET_COLUMN].dropna().unique()).issubset(set(mapping.keys())):
        df[TARGET_COLUMN] = df[TARGET_COLUMN].map(mapping)

# Split into X / y
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X.shape, y.shape

In [ ]:
# =========================
# 5) Identify categorical vs numerical columns
# =========================
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

print("Categorical columns:", len(categorical_cols))
print(categorical_cols)

print("\nNumerical columns:", len(numerical_cols))
print(numerical_cols)

## 6) Train/Test split

Use `stratify=y` when the task is classification and `y` is categorical/binary.

In [ ]:
# =========================
# 6) Train/Test split
# =========================
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y if y.nunique() <= 20 else None  # simple heuristic
)

X_train_raw.shape, X_test_raw.shape

## 7) Build preprocessing pipeline

Typical approach:
- **Categorical**: impute missing values + OneHotEncoder
- **Numerical**: impute missing values + (optional) scaling

If your model doesn't need scaling (e.g., tree-based models), you can remove `StandardScaler()`.

In [ ]:
# =========================
# 7) Preprocessing pipeline
# =========================
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),  # optional for some models
])

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_pipeline, categorical_cols),
        ("num", numerical_pipeline, numerical_cols),
    ],
    remainder="drop"
)

preprocessor

In [ ]:
# =========================
# 8) Fit on train, transform train and test
# =========================
X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

In [ ]:
# =========================
# 9) Get feature names (useful for debugging/importance)
# =========================
feature_names = []
try:
    cat_feature_names = preprocessor.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_cols)
    num_feature_names = numerical_cols
    feature_names = list(cat_feature_names) + list(num_feature_names)
    print("Total feature names:", len(feature_names))
    print("Example:", feature_names[:20])
except Exception as e:
    print("Could not extract feature names:", e)

## 10) Save artifacts

Common artifacts:
- `(X_train, y_train)` and `(X_test, y_test)` as pickle
- `preprocessor` (so predict-time uses the same transformations)
- `feature_names` (optional)

You can choose where to store them (e.g., `data/processed/` and `models/`).

In [ ]:
# =========================
# 10) Save processed data + preprocessor
# =========================
train_artifact_path = PROCESSED_DIR / "train_data.pkl"
test_artifact_path = PROCESSED_DIR / "test_data.pkl"
preprocessor_path = MODELS_DIR / "preprocessor.pkl"
feature_names_path = MODELS_DIR / "feature_names.pkl"

with open(train_artifact_path, "wb") as f:
    pickle.dump((X_train, y_train), f)

with open(test_artifact_path, "wb") as f:
    pickle.dump((X_test, y_test), f)

with open(preprocessor_path, "wb") as f:
    pickle.dump(preprocessor, f)

with open(feature_names_path, "wb") as f:
    pickle.dump(feature_names, f)

train_artifact_path, test_artifact_path, preprocessor_path, feature_names_path

## 11) Sanity check: reload artifacts

This is a quick check to confirm what you saved can be loaded and used later.

In [ ]:
# =========================
# 11) Reload sanity check
# =========================
with open(train_artifact_path, "rb") as f:
    X_train_loaded, y_train_loaded = pickle.load(f)

with open(preprocessor_path, "rb") as f:
    preprocessor_loaded = pickle.load(f)

print("Loaded X_train shape:", X_train_loaded.shape)
print("Loaded y_train shape:", y_train_loaded.shape)
print("Loaded preprocessor type:", type(preprocessor_loaded))

## Notes / next steps

- Add dataset-specific cleaning rules (outliers, parsing dates, dropping leakage columns, etc.).
- Consider validating schema / column presence before processing.
- If you need a validation set, split `X_train_raw` into train/val.
- For deployment/prediction, always apply the saved `preprocessor` before calling `model.predict`.